# Bin packing problem: the fewest bins that hold every item

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ProvenDecisions/optimization-examples/blob/main/notebooks/04-integer-programming/bin-packing.ipynb)

Solved and checked by [ProvenDecisions](https://proven-decisions.ai/examples/bin-packing?ref=github): the step-by-step walkthrough, the dual or KKT certificate and a version you can change and re-solve are on the example's page.

## The problem

Pack items of sizes 4, 8, 1, 4, 2, 1 into bins of capacity 10 using as few bins as possible.

## The model

Each line has a plain-language note.

$$
\begin{aligned}\min\;& y_{1} + y_{2} + y_{3} &&\text{make this as small as possible}\\ \text{s.t.}\;& x_{11} + x_{12} + x_{13} = 1 &&\text{limit 1: left side must equal 1}\\ \;& x_{21} + x_{22} + x_{23} = 1 &&\text{limit 2: left side must equal 1}\\ \;& x_{31} + x_{32} + x_{33} = 1 &&\text{limit 3: left side must equal 1}\\ \;& x_{41} + x_{42} + x_{43} = 1 &&\text{limit 4: left side must equal 1}\\ \;& x_{51} + x_{52} + x_{53} = 1 &&\text{limit 5: left side must equal 1}\\ \;& x_{61} + x_{62} + x_{63} = 1 &&\text{limit 6: left side must equal 1}\\ \;& 4 x_{11} + 8 x_{21} + x_{31} + 4 x_{41} + 2 x_{51} + x_{61} \le 10 y_{1} &&\text{limit 7: left side cannot exceed the right side}\\ \;& 4 x_{12} + 8 x_{22} + x_{32} + 4 x_{42} + 2 x_{52} + x_{62} \le 10 y_{2} &&\text{limit 8: left side cannot exceed the right side}\\ \;& 4 x_{13} + 8 x_{23} + x_{33} + 4 x_{43} + 2 x_{53} + x_{63} \le 10 y_{3} &&\text{limit 9: left side cannot exceed the right side}\\ &x_{11}, x_{12}, x_{13}, x_{21}, x_{22}, x_{23}, x_{31}, x_{32}, x_{33}, x_{41}, x_{42}, x_{43}, x_{51}, x_{52}, x_{53}, x_{61}, x_{62}, x_{63}, y_{1}, y_{2}, y_{3} \in \{0,1\} &&\text{yes (1) or no (0)}\end{aligned}
$$

## Solve it with cvxpy

Run the cell (in Colab, `cvxpy` is installed by the first line).

In [ ]:
%pip install -q cvxpy highspy

In [1]:
import cvxpy as cp

x11 = cp.Variable(name='x11', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 1, j = 1)
x12 = cp.Variable(name='x12', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 1, j = 2)
x13 = cp.Variable(name='x13', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 1, j = 3)
x21 = cp.Variable(name='x21', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 2, j = 1)
x22 = cp.Variable(name='x22', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 2, j = 2)
x23 = cp.Variable(name='x23', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 2, j = 3)
x31 = cp.Variable(name='x31', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 3, j = 1)
x32 = cp.Variable(name='x32', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 3, j = 2)
x33 = cp.Variable(name='x33', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 3, j = 3)
x41 = cp.Variable(name='x41', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 4, j = 1)
x42 = cp.Variable(name='x42', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 4, j = 2)
x43 = cp.Variable(name='x43', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 4, j = 3)
x51 = cp.Variable(name='x51', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 5, j = 1)
x52 = cp.Variable(name='x52', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 5, j = 2)
x53 = cp.Variable(name='x53', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 5, j = 3)
x61 = cp.Variable(name='x61', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 6, j = 1)
x62 = cp.Variable(name='x62', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 6, j = 2)
x63 = cp.Variable(name='x63', boolean=True)  # 1 if item i (sizes 4, 8, 1, 4, 2, 1 for i=1..6) is placed in bin j (yes/no) (i = 6, j = 3)
y1 = cp.Variable(name='y1', boolean=True)  # 1 if bin j is used (yes/no) (j = 1)
y2 = cp.Variable(name='y2', boolean=True)  # 1 if bin j is used (yes/no) (j = 2)
y3 = cp.Variable(name='y3', boolean=True)  # 1 if bin j is used (yes/no) (j = 3)

objective = cp.Minimize(y1 + y2 + y3)
constraints = [
    x11 + x12 + x13 == 1,
    x21 + x22 + x23 == 1,
    x31 + x32 + x33 == 1,
    x41 + x42 + x43 == 1,
    x51 + x52 + x53 == 1,
    x61 + x62 + x63 == 1,
    4*x11 + 8*x21 + x31 + 4*x41 + 2*x51 + x61 <= 10*y1,
    4*x12 + 8*x22 + x32 + 4*x42 + 2*x52 + x62 <= 10*y2,
    4*x13 + 8*x23 + x33 + 4*x43 + 2*x53 + x63 <= 10*y3,
]
problem = cp.Problem(objective, constraints)
problem.solve(solver='HIGHS')
print(problem.status, problem.value)
print('x11 =', x11.value)
print('x12 =', x12.value)
print('x13 =', x13.value)
print('x21 =', x21.value)
print('x22 =', x22.value)
print('x23 =', x23.value)
print('x31 =', x31.value)
print('x32 =', x32.value)
print('x33 =', x33.value)
print('x41 =', x41.value)
print('x42 =', x42.value)
print('x43 =', x43.value)
print('x51 =', x51.value)
print('x52 =', x52.value)
print('x53 =', x53.value)
print('x61 =', x61.value)
print('x62 =', x62.value)
print('x63 =', x63.value)
print('y1 =', y1.value)
print('y2 =', y2.value)
print('y3 =', y3.value)

optimal 2.0
x11 = 1.0
x12 = -0.0
x13 = -0.0
x21 = -0.0
x22 = -0.0
x23 = 1.0
x31 = 1.0
x32 = -0.0
x33 = -0.0
x41 = 1.0
x42 = -0.0
x43 = -0.0
x51 = -0.0
x52 = -0.0
x53 = 1.0
x61 = 1.0
x62 = -0.0
x63 = -0.0
y1 = 1.0
y2 = -0.0
y3 = 1.0


## The checked answer

The smallest objective value is **2**, at x11 = 1, x12 = 0, x13 = 0, x21 = 0, x22 = 0, x23 = 1, x31 = 1, x32 = 0, x33 = 0, x41 = 1, x42 = 0, x43 = 0, x51 = 0, x52 = 0, x53 = 1, x61 = 1, x62 = 0, x63 = 0, y1 = 1, y2 = 0, y3 = 1. All 9 constraints are satisfied (violation ≤0e+00). **Optimal within gap tolerance.**

**Problem class:** [ILP](https://proven-decisions.ai/problem-classes?ref=github); **solver:** HIGHS. What makes the answer proven: a [dual or KKT certificate](https://proven-decisions.ai/glossary?ref=github), explained in plain words.

Step-by-step walkthrough, the proof of optimality and more: [Bin packing problem: the fewest bins that hold every item](https://proven-decisions.ai/examples/bin-packing?ref=github)

**Change the numbers and solve it free:** [this model in the ProvenDecisions solver](https://proven-decisions.ai/?try=minimize%20y1%20%2B%20y2%20%2B%20y3%0Asubject%20to%0Ax11%20%2B%20x12%20%2B%20x13%20%3D%201%0Ax21%20%2B%20x22%20%2B%20x23%20%3D%201%0Ax31%20%2B%20x32%20%2B%20x33%20%3D%201%0Ax41%20%2B%20x42%20%2B%20x43%20%3D%201%0Ax51%20%2B%20x52%20%2B%20x53%20%3D%201%0Ax61%20%2B%20x62%20%2B%20x63%20%3D%201%0A4%20x11%20%2B%208%20x21%20%2B%201%20x31%20%2B%204%20x41%20%2B%202%20x51%20%2B%201%20x61%20%3C%3D%2010%20y1%0A4%20x12%20%2B%208%20x22%20%2B%201%20x32%20%2B%204%20x42%20%2B%202%20x52%20%2B%201%20x62%20%3C%3D%2010%20y2%0A4%20x13%20%2B%208%20x23%20%2B%201%20x33%20%2B%204%20x43%20%2B%202%20x53%20%2B%201%20x63%20%3C%3D%2010%20y3%0Abinary%3A%20x11%2C%20x12%2C%20x13%2C%20x21%2C%20x22%2C%20x23%2C%20x31%2C%20x32%2C%20x33%2C%20x41%2C%20x42%2C%20x43%2C%20x51%2C%20x52%2C%20x53%2C%20x61%2C%20x62%2C%20x63%2C%20y1%2C%20y2%2C%20y3%0A%23%20xij%20%3D%201%20if%20item%20i%20%28sizes%204%2C%208%2C%201%2C%204%2C%202%2C%201%20for%20i%3D1..6%29%20is%20placed%20in%20bin%20j%20%28yes/no%29%0A%23%20yj%20%3D%201%20if%20bin%20j%20is%20used%20%28yes/no%29&ref=github)